In [1]:
# =========================
# 1) Imports + file path
# =========================
import geopandas as gpd
import pandas as pd
import fiona

GPKG_PATH = r"D:\M2_MoSEF\Acclim\DataCollection\data\NUTS_RG_20M_2024_4326.gpkg"  # <-- adjust if needed

# Optional display settings
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("GeoPackage path:", GPKG_PATH)

GeoPackage path: D:\M2_MoSEF\Acclim\DataCollection\data\NUTS_RG_20M_2024_4326.gpkg


In [2]:
# =========================
# 2) List layers in the GeoPackage
# =========================
layers = fiona.listlayers(GPKG_PATH)
print("Layers found:")
for i, lyr in enumerate(layers, 1):
    print(f"{i:02d}. {lyr}")

Layers found:
01. NUTS_RG_20M_2024_4326.gpkg


In [3]:
# =========================
# 3) Load the first/target layer
#    (Usually there is one main layer, but let's inspect safely)
# =========================
# If you see the layer name from previous cell, put it here explicitly.
# Example: layer_name = "NUTS_RG_20M_2024_4326"
layer_name = layers[0]

gdf = gpd.read_file(GPKG_PATH, layer=layer_name)

print("Loaded layer:", layer_name)
print("Shape:", gdf.shape)
print("CRS:", gdf.crs)
print("\nColumns:")
print(gdf.columns.tolist())
gdf.head(3)

Loaded layer: NUTS_RG_20M_2024_4326.gpkg
Shape: (1798, 9)
CRS: EPSG:4326

Columns:
['NUTS_ID', 'LEVL_CODE', 'CNTR_CODE', 'NAME_LATN', 'NUTS_NAME', 'MOUNT_TYPE', 'URBN_TYPE', 'COAST_TYPE', 'geometry']


,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,geometry
0,AL011,3,AL,Dibër,Dibër,NaN,NaN,NaN,"POLYGON ((20.2208 41.86937, 20.56287 41.84614,..."
1,AL012,3,AL,Durrës,Durrës,NaN,NaN,NaN,"POLYGON ((19.55733 41.57767, 19.83218 41.56823..."
2,AL013,3,AL,Kukës,Kukës,NaN,NaN,NaN,"POLYGON ((20.59429 41.87733, 20.56287 41.84614..."


In [4]:
# =========================
# 4) Quick column diagnostics (important for code mapping)
# =========================
# We expect columns like: CNTR_CODE, NUTS_ID, LEVL_CODE, NAME_LATN (or NAME_ENGL), geometry

for col in ["CNTR_CODE", "NUTS_ID", "LEVL_CODE", "NAME_LATN", "NAME_ENGL"]:
    print(f"{col}: {'YES' if col in gdf.columns else 'NO'}")

print("\nDtypes:")
print(gdf.dtypes)


CNTR_CODE: YES
NUTS_ID: YES
LEVL_CODE: YES
NAME_LATN: YES
NAME_ENGL: NO

Dtypes:
NUTS_ID         object
LEVL_CODE        int16
CNTR_CODE       object
NAME_LATN       object
NUTS_NAME       object
MOUNT_TYPE     float64
URBN_TYPE      float64
COAST_TYPE     float64
geometry      geometry
dtype: object


In [5]:
# =========================
# 6) Filter target countries + NUTS2/NUTS3
# =========================
TARGET_CNTR = {"FR", "BE", "IT", "LU"}

gdf4 = gdf[gdf["CNTR_CODE"].isin(TARGET_CNTR)].copy()
gdf4 = gdf4[gdf4["LEVL_CODE"].isin([2, 3])].copy()

print("Filtered shape (FR/BE/IT/LU, NUTS2/NUTS3):", gdf4.shape)
print("\nCounts by country and level:")
print(
    gdf4.groupby(["CNTR_CODE", "LEVL_CODE"])
        .size()
        .rename("n")
        .reset_index()
        .sort_values(["CNTR_CODE", "LEVL_CODE"])
)

# Display a sample
cols_to_show = [c for c in ["CNTR_CODE", "NUTS_ID", "LEVL_CODE", "NAME_LATN", "NAME_ENGL"] if c in gdf4.columns]
gdf4[cols_to_show].head(20)

Filtered shape (FR/BE/IT/LU, NUTS2/NUTS3): (313, 9)

Counts by country and level:
  CNTR_CODE  LEVL_CODE    n
0        BE          2   11
1        BE          3   44
2        FR          2   27
3        FR          3  101
4        IT          2   21
5        IT          3  107
6        LU          2    1
7        LU          3    1


,CNTR_CODE,NUTS_ID,LEVL_CODE,NAME_LATN
47,BE,BE100,3,Arr. de Bruxelles-Capitale/Arr. Brussel-Hoofdstad
48,BE,BE211,3,Arr. Antwerpen
49,BE,BE212,3,Arr. Mechelen
50,BE,BE213,3,Arr. Turnhout
51,BE,BE223,3,Arr. Tongeren
52,BE,BE224,3,Arr. Hasselt
53,BE,BE225,3,Arr. Maaseik
54,BE,BE231,3,Arr. Aalst
55,BE,BE232,3,Arr. Dendermonde
56,BE,BE233,3,Arr. Eeklo


In [6]:
# =========================
# 7) Build standardized NUTS2 / NUTS3 tables (preview)
# =========================
name_col = "NAME_LATN" if "NAME_LATN" in gdf4.columns else ("NAME_ENGL" if "NAME_ENGL" in gdf4.columns else None)
if name_col is None:
    raise KeyError("Neither NAME_LATN nor NAME_ENGL found in the layer.")

country_map = {
    "FR": "France",
    "BE": "Belgium",
    "IT": "Italy",
    "LU": "Luxembourg",
}

gdf4["country"] = gdf4["CNTR_CODE"].map(country_map)

regions_nuts2 = (
    gdf4[gdf4["LEVL_CODE"] == 2]
    .copy()
    .rename(columns={
        "CNTR_CODE": "country_code",
        "NUTS_ID": "region_code",
        name_col: "region_name",
    })
)

departments_nuts3 = (
    gdf4[gdf4["LEVL_CODE"] == 3]
    .copy()
    .rename(columns={
        "CNTR_CODE": "country_code",
        "NUTS_ID": "dept_code",
        name_col: "dept_name",
    })
)

print("regions_nuts2:", regions_nuts2.shape)
print("departments_nuts3:", departments_nuts3.shape)

display_cols_r = [c for c in ["country", "country_code", "region_code", "region_name"] if c in regions_nuts2.columns]
display_cols_d = [c for c in ["country", "country_code", "dept_code", "dept_name"] if c in departments_nuts3.columns]

print("\nSample regions_nuts2")
display(regions_nuts2[display_cols_r].head(10))

print("\nSample departments_nuts3")
display(departments_nuts3[display_cols_d].head(10))


regions_nuts2: (60, 10)
departments_nuts3: (253, 10)

Sample regions_nuts2


,country,country_code,region_code,region_name
81,Italy,IT,ITC4,Lombardia
82,Italy,IT,ITF1,Abruzzo
83,Italy,IT,ITF2,Molise
84,Italy,IT,ITF3,Campania
85,Italy,IT,ITF4,Puglia
86,Italy,IT,ITF5,Basilicata
87,Italy,IT,ITF6,Calabria
88,Italy,IT,ITG1,Sicilia
89,Italy,IT,ITG2,Sardegna
90,Italy,IT,ITH1,Provincia Autonoma di Bolzano/Bozen



Sample departments_nuts3


,country,country_code,dept_code,dept_name
47,Belgium,BE,BE100,Arr. de Bruxelles-Capitale/Arr. Brussel-Hoofdstad
48,Belgium,BE,BE211,Arr. Antwerpen
49,Belgium,BE,BE212,Arr. Mechelen
50,Belgium,BE,BE213,Arr. Turnhout
51,Belgium,BE,BE223,Arr. Tongeren
52,Belgium,BE,BE224,Arr. Hasselt
53,Belgium,BE,BE225,Arr. Maaseik
54,Belgium,BE,BE231,Arr. Aalst
55,Belgium,BE,BE232,Arr. Dendermonde
56,Belgium,BE,BE233,Arr. Eeklo
